# EchoMatch on PARTIALSMAL

This notebook demonstrates **EchoMatch** (Xie et al., CVPR 2025) —  
a partial-to-partial shape matching method — using **geomfum** and **benchfum**.

We cover:
1. Building `EchoMatchNet` from the JSON config
2. Loading the PARTIALSMAL dataset (`PartialSmalPairsDataset`)
3. Running inference on a test pair
4. Inspecting overlap scores and functional map
5. Evaluating `OverlapIoU` and `PartialGeodesicError` over the test set
6. Training from scratch on PARTIALSMAL

In [ ]:
import os

# Must be set BEFORE any geomfum / geomstats imports
os.environ["GEOMSTATS_BACKEND"] = "pytorch"

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

## 1. Dataset — PARTIALSMAL

`PartialSmalPairsDataset` loads the PARTIALSMAL dataset.  
Directory layout:
```
PARTIALSMAL/
  train/
    shapes/   <- .off partial meshes (e.g. cuts_0_cow_02.off)
    maps/     <- .vts pair files (e.g. cuts_0_cow_02_cuts_22_dog_05.vts)
  test/
    shapes/
    maps/
```

Each `.vts` file contains one 1-indexed integer per line:  
line `j` is the vertex in shape X matching vertex `j` of shape Y (or `-1` for none).

In [ ]:
import sys

sys.path.append("../../../geomfum/")

from geomfum.dataset.partial import PartialSmalPairsDataset

# Use the dummy dataset for a quick smoke test.
# Set USE_DUMMY = False and point PARTIALSMAL_ROOT to the full dataset for real experiments.
USE_DUMMY = True

if USE_DUMMY:
    PARTIALSMAL_ROOT = "../../../datasets/dummy_datasets/dummy_partial_smal"
else:
    PARTIALSMAL_ROOT = "../../../datasets/PARTIALSMAL"

K_EIG = 128  # number of LBO eigenvectors

test_dataset = PartialSmalPairsDataset(
    PARTIALSMAL_ROOT,
    split="test",
    spectral=True,
    k=K_EIG,
    device=DEVICE,
)

train_dataset = PartialSmalPairsDataset(
    PARTIALSMAL_ROOT,
    split="train",
    spectral=True,
    k=K_EIG,
    device=DEVICE,
)

print(f"Train pairs: {len(train_dataset)},  Test pairs: {len(test_dataset)}")

# Inspect one pair
pair = test_dataset[0]
shape_a = pair["source"]["shape"]
shape_b = pair["target"]["shape"]
mask_a = pair["source"]["mask"]  # Tensor[n_a], 1 = in overlap
mask_b = pair["target"]["mask"]  # Tensor[n_b]
corr_a = pair["source"]["corr"]  # Tensor[n_valid] — GT vertex indices in A
corr_b = pair["target"]["corr"]  # Tensor[n_valid] — paired GT vertices in B

print(f"Shape A: {shape_a.n_vertices} vertices,  mask coverage: {mask_a.mean():.2%}")
print(f"Shape B: {shape_b.n_vertices} vertices,  mask coverage: {mask_b.mean():.2%}")
print(f"GT correspondences: {len(corr_a)}")

c:\Users\giuli\OneDrive\Research\geomfum_proj\venv\Lib\site-packages\gsops\pytorch\sparse.py:21: UserWarning: Sparse CSC tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\SparseCsrTensorImpl.cpp:55.)
  return _torch.sparse_csc_tensor(ccol_indices, row_indices, values, size=array.shape)


Train pairs: 3,  Test pairs: 3
Shape A: 2843 vertices,  mask coverage: 84.91%
Shape B: 2620 vertices,  mask coverage: 58.21%
GT correspondences: 2414


## 2. Build `EchoMatchNet` from JSON config

The config `benchfum/configs/models/echo_match.json` encodes all EchoMatch hyperparameters:

| Component | Setting |
|---|---|
| Feature extractor | DiffusionNet, k=128 in/out=128, WKS 128-dim |
| Functional map | λ=100, resolvent γ=0.5, bijective |
| Permutation net | SoftmaxNeighborFinder τ=0.10 |
| Echo scorer | neighbor_size=64 |
| Overlap refiner | small DiffusionNet 1→1, hidden=16, 3 blocks |

In [ ]:
from benchfum._build import build_model_from_json

MODEL_CONFIG = "../../../geomfum/benchfum/configs/models/echo_match.json"

model = build_model_from_json(MODEL_CONFIG, device=DEVICE)
print(model)

ValueError: Unknown component type: 'EchoMatchNet'. Available types: ['AdjointBijectiveZoomOut', 'ArangeSubsampler', 'CorrespondenceRefinementPipeline', 'DescriptorPipeline', 'DistanceFromLandmarksDescriptor', 'FeatureMatcher', 'FunctionalMapMatcher', 'FunctionalMapOptimizer', 'HeatKernelSignature', 'IcpRefiner', 'IdentityRefiner', 'L2InnerNormalizer', 'LBFactorBuilder', 'LandmarkHeatKernelSignature', 'LandmarkWaveKernelSignature', 'MultFactorBuilder', 'NeighborFinder', 'NeuralZoomOut', 'NormalizedDescriptor', 'OrientFactorBuilder', 'OrthogonalRefiner', 'P2pFromFmConverter', 'RefinementPipeline', 'SDPFactorBuilder', 'ShotDescriptor', 'SoftmaxNeighborFinder', 'UrrsmHksDomain', 'UrrsmWksDomain', 'WaveKernelSignature', 'ZoomOut']

## 3. Inference on a test pair

`EchoMatchNet.forward` returns a `CorrespondenceResult` with extra fields:
- `fmap12`, `fmap21` — functional maps in both directions
- `p2p21` — point-to-point map (only in eval mode)
- `overlap_ab` — per-vertex overlap scores for shape A ∈ [0, 1]
- `overlap_ba` — per-vertex overlap scores for shape B

In [ ]:
from geomfum.learning.wrappers import TrainedModelWrapper

matcher = TrainedModelWrapper(model, device=DEVICE)

# overlap scores are always computed (both directions); p2p12/fmap21 need bidirectional=True
result = matcher(shape_a, shape_b)

print("fmap12:     ", result.fmap12.shape)      # [K_b, K_a]
print("p2p21:      ", result.p2p21.shape)        # [n_b]
print("overlap_ab: ", result.overlap_ab.shape)   # [n_a]
print("overlap_ba: ", result.overlap_ba.shape)   # [n_b]

overlap_ab = result.overlap_ab.cpu()
overlap_ba = result.overlap_ba.cpu()
print(f"\nPredicted overlap A: {(overlap_ab > 0.5).float().mean():.2%} of vertices")
print(f"GT        overlap A: {mask_a.cpu().mean():.2%} of vertices")
print(f"\nPredicted overlap B: {(overlap_ba > 0.5).float().mean():.2%} of vertices")
print(f"GT        overlap B: {mask_b.cpu().mean():.2%} of vertices")

## 4. Overlap IoU on the test pair

`OverlapIoU` computes `1 - IoU` (lower = better overlap prediction).  
We display `IoU` directly for readability.

In [ ]:
from geomfum.learning.losses import OverlapIoU

iou_metric = OverlapIoU()

# iou_metric returns 1 - IoU  (a loss; 0 = perfect)
loss_ab = iou_metric(overlap_ab.to(DEVICE), mask_a.to(DEVICE))
loss_ba = iou_metric(overlap_ba.to(DEVICE), mask_b.to(DEVICE))

print(f"Overlap IoU  (shape A): {1 - loss_ab.item():.4f}")
print(f"Overlap IoU  (shape B): {1 - loss_ba.item():.4f}")

## 5. Full evaluation on the test set

We iterate over all test pairs and compute mean `OverlapIoU`.  
(Geodesic error requires pre-computed distance matrices; omitted here for speed.)

In [ ]:
import numpy as np
from tqdm import tqdm

iou_metric = OverlapIoU()
iou_vals_a, iou_vals_b = [], []

for pair in tqdm(test_dataset, desc="Evaluating"):
    sa = pair["source"]["shape"]
    sb = pair["target"]["shape"]
    ma = pair["source"]["mask"]
    mb = pair["target"]["mask"]

    try:
        res = matcher(sa, sb)
    except Exception as e:
        print(f"  [warn] pair failed: {e}")
        continue

    ov_a = res.overlap_ab.to(DEVICE)
    ov_b = res.overlap_ba.to(DEVICE)

    iou_vals_a.append(1.0 - iou_metric(ov_a, ma).item())
    iou_vals_b.append(1.0 - iou_metric(ov_b, mb).item())

print(f"Mean Overlap IoU (shape A): {np.mean(iou_vals_a):.4f}")
print(f"Mean Overlap IoU (shape B): {np.mean(iou_vals_b):.4f}")
print(f"Mean Overlap IoU (avg):     {np.mean(iou_vals_a + iou_vals_b):.4f}")

## 6. Training from scratch on PARTIALSMAL

The training config `benchfum/configs/training/echo_match.json` encodes:

| Setting | Value |
|---|---|
| Optimizer | Adam lr=1e-3, betas=(0.9, 0.99) |
| Scheduler | StepLR step=100, γ=0.5 |
| Epochs | 300 |
| Grad clip | 1.0 |
| Train losses | OrthonormalityLoss + BijectivityLoss + WeightedBCELoss + CrossNCELoss + SelfNCELoss |
| Val losses | PartialGeodesicError + OverlapIoU + PCKMetric |

In [ ]:
from benchfum._build import build_trainer_from_json

TRAINING_CONFIG = "../../../geomfum/benchfum/configs/training/echo_match.json"
CHECKPOINT_SAVE = "./echo_match_partialsmal_best.pth"

# Fresh model for training
model_train = build_model_from_json(MODEL_CONFIG, device=DEVICE)

trainer = build_trainer_from_json(
    TRAINING_CONFIG,
    model=model_train,
    train_set=train_dataset,
    val_set=test_dataset,
)
trainer.checkpoint_path = CHECKPOINT_SAVE
trainer.device = DEVICE

print("Trainer config:")
print(f"  epochs       = {trainer.epochs}")
print(f"  optimizer    = {type(trainer.optimizer).__name__}")
print(f"  scheduler    = {type(trainer.scheduler).__name__}")
print(f"  grad_clip    = {trainer.grad_clip_norm}")
print(f"  monitor      = {trainer.monitor_metric} ({trainer.mode})")
print(
    f"  train losses = {[type(l).__name__ for l in trainer.train_loss_manager.losses]}"
)
print(f"  val   losses = {[type(l).__name__ for l in trainer.val_loss_manager.losses]}")

In [ ]:
# Smoke test: run just 2 epochs on the dummy dataset to verify the pipeline
# For real training use trainer.epochs = 300 (or as set in the config) and full dataset
if USE_DUMMY:
    trainer.epochs = 2
    trainer.train()
else:
    # Uncomment for full training on PARTIALSMAL (300 epochs, ~hours on GPU)
    # trainer.train()
    pass

## 7. Load a saved checkpoint and evaluate

After training (or if you already have a checkpoint), load it via `TrainedModelWrapper`.

In [ ]:
import os

EVAL_CHECKPOINT = CHECKPOINT_SAVE  # point to your checkpoint here

if os.path.exists(EVAL_CHECKPOINT):
    eval_model = build_model_from_json(MODEL_CONFIG, device=DEVICE)
    eval_matcher = TrainedModelWrapper(
        eval_model, device=DEVICE, checkpoint_path=EVAL_CHECKPOINT
    )
    print("Checkpoint loaded — ready to evaluate.")
else:
    print(f"No checkpoint at {EVAL_CHECKPOINT}. Run trainer.train() first.")

## 8. Run the benchfum challenge runner

The full benchmark can also be run from the command line.

**Smoke test on the dummy dataset** (fast, for verifying the pipeline):

```bash
cd geomfum/
GEOMSTATS_BACKEND=pytorch python -m benchfum.challenges.partial_shape.run \
    --config benchfum/configs/benchmarks/partial/partialsmal_partial.json \
    --dataset ../datasets/dummy_datasets/dummy_partial_smal
```

**Full train + evaluate on PARTIALSMAL**:

```bash
GEOMSTATS_BACKEND=pytorch python -m benchfum.challenges.partial_shape.run \
    --config benchfum/configs/benchmarks/partial/partialsmal_partial.json \
    --train \
    --train_dataset ../datasets/PARTIALSMAL \
    --val_dataset   ../datasets/PARTIALSMAL
```

**Evaluate only (with a checkpoint)**:

```bash
GEOMSTATS_BACKEND=pytorch python -m benchfum.challenges.partial_shape.run \
    --config benchfum/configs/benchmarks/partial/partialsmal_partial.json \
    --dataset ../datasets/PARTIALSMAL
```